# RotQuant ternary study · version 1

Compare **FP16**, fresh **W6/W6, W5/W6 and W4/W6** controls, and
**ternary weights** before and after quantization-aware training (QAT).
Include the official **Bonsai2-4B** model as a measured comparison.
The study records held-out quality, public task scores, complete model
bytes, and prefill/decode speed. This is an open research recipe;
PrismML's private training method is not reproduced here.

Select an **A100 40 GB-class GPU** and high host RAM. Choose **Run all**
and upload the supplied source ZIP. The separate W4 notebook stays
available. This notebook has no saved model results; the default
16 examples per public benchmark are a pilot.

## Settings
Keep the same source ZIP and settings to resume. Logs and completed
checkpoints stay on Drive. The eight-hour maximum is divided across
seeds; setup and idle GPU time are additional. The run stops at its
budget, so a cold build or interrupted session may leave partial results.

To continue a stopped study with corrected source, set `RESUME_FROM` to its
Drive result folder and select its single seed. Verified completed results
are retained in a separate child run, with their original provenance.
Set `BUDGET_MINUTES` to the remaining allowance, at most the parent's
unused active execution minutes. Set `FRESH_BUDGET = True` to explicitly
start a new allowance of `BUDGET_MINUTES` (up to 480), preserving the
parent's consumed time in the continuation receipt.
When continuing after a parity correction,
set `PARITY_CORRECTION` to the saved diagnostic receipt; its checksum
is recorded in the child identity.

In [ ]:
from pathlib import Path
import hashlib, inspect, json, math, os, re, shutil, stat, subprocess, sys, tempfile, zipfile
from pathlib import PurePosixPath

STUDY_VERSION = "ternary-v1"
SOURCE_MODE = "upload"  # "git" requires an explicitly published full commit below.
SOURCE_ARCHIVE = ""  # Optional: paste the saved Drive ZIP path to reuse it.
GIT_COMMIT = ""  # Exactly 40 hexadecimal characters; branches such as main are rejected.
RESUME_FROM = ""  # Optional: parent Drive result folder; leave empty for a fresh study.
FRESH_BUDGET = False  # Explicitly start a new continuation allowance.
PARITY_CORRECTION = ""  # Optional: saved parity diagnostic receipt for this continuation.
SEEDS = [0]
PUBLIC_SAMPLES = 16  # Raise before the run for a larger independent task comparison.
BUDGET_MINUTES = 480
RUN_EXPERIMENT = True
REQUIRE_FAST_HADAMARD = True
USE_HF_SECRET = False
RESULT_BASE = Path("/content/drive/MyDrive/rotquant/ternary_research_v1")
LOCAL_BASE = Path("/content/rotquant-ternary-research-v1")
UV_VERSION = "0.9.5"

assert SOURCE_MODE in {"upload", "git"}
assert SEEDS and len(set(SEEDS)) == len(SEEDS) and all(type(seed) is int and seed >= 0 for seed in SEEDS)
assert type(BUDGET_MINUTES) is int and len(SEEDS) <= BUDGET_MINUTES <= 480
assert type(PUBLIC_SAMPLES) is int and 1 <= PUBLIC_SAMPLES <= 541
assert not RESUME_FROM or len(SEEDS) == 1, "A continuation requires one seed."
assert type(FRESH_BUDGET) is bool and (not FRESH_BUDGET or RESUME_FROM), "FRESH_BUDGET requires RESUME_FROM."
assert not PARITY_CORRECTION or RESUME_FROM, "A parity correction requires RESUME_FROM."

## Load the code
Upload the supplied ZIP, or enter its saved Drive path above. Every source
file is checked against the bundle manifest, including on resume. A changed
source or configuration gets a separate result directory.

In [ ]:
def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def safe_member_name(name):
    path = PurePosixPath(name)
    if (not name or path.is_absolute() or ".." in path.parts or "\\" in name
            or any(part in {".git", ".venv"} for part in path.parts)):
        raise ValueError(f"Unsafe source archive member: {name!r}")
    return path


def verify_source_bundle(repository):
    repository = Path(repository)
    if any(path.is_symlink() for path in repository.rglob("*")):
        raise ValueError("Source tree must not contain symbolic links")
    manifest_path = repository / "SOURCE_BUNDLE.json"
    manifest = json.loads(manifest_path.read_text())
    if manifest.get("protocol") != "rotquant-source-bundle-v1":
        raise ValueError("Unrecognized source bundle manifest")
    entries = manifest.get("files_sha256")
    if not isinstance(entries, dict) or not entries:
        raise ValueError("The source bundle must list its file checksums")
    for name, expected in entries.items():
        path = repository / safe_member_name(name)
        if not path.is_file() or path.is_symlink() or file_sha256(path) != expected:
            raise ValueError(f"Source checksum mismatch: {name}")
    expected_files = set(entries) | {"SOURCE_BUNDLE.json"}
    actual_files = {str(path.relative_to(repository)) for path in repository.rglob("*")
                    if path.is_file() and "__pycache__" not in path.parts
                    and not any(part.endswith(".egg-info") for part in path.parts)}
    if actual_files != expected_files:
        raise ValueError("Source tree has unlisted files; use a fresh source directory")
    return file_sha256(manifest_path)


def unpack_source_bundle(archive_path, cache_root):
    archive_path, cache_root = Path(archive_path), Path(cache_root)
    archive_hash = file_sha256(archive_path)
    repository = cache_root / archive_hash
    if repository.exists():
        return repository, verify_source_bundle(repository)
    cache_root.mkdir(parents=True, exist_ok=True)
    with tempfile.TemporaryDirectory(prefix="extract-", dir=cache_root) as temporary:
        target = Path(temporary) / "repository"
        target.mkdir()
        with zipfile.ZipFile(archive_path) as archive:
            members = archive.infolist()
            if len(members) > 50000 or sum(item.file_size for item in members) > 512 * 2**20:
                raise ValueError("Source archive exceeds the 512 MiB extraction limit")
            seen = set()
            for item in members:
                name = safe_member_name(item.filename)
                mode = stat.S_IFMT(item.external_attr >> 16)
                if str(name) in seen or mode not in (0, stat.S_IFREG, stat.S_IFDIR):
                    raise ValueError("Duplicate or non-regular source archive entry")
                seen.add(str(name))
                destination = target / name
                if item.is_dir():
                    destination.mkdir(parents=True, exist_ok=True)
                    continue
                destination.parent.mkdir(parents=True, exist_ok=True)
                with archive.open(item) as source, destination.open("xb") as sink:
                    shutil.copyfileobj(source, sink)
        identity = verify_source_bundle(target)
        target.rename(repository)
    return repository, identity

In [ ]:
from google.colab import drive, files
import torch
from importlib import metadata
assert torch.cuda.is_available(), "Choose a GPU runtime."
assert torch.cuda.get_device_properties(0).total_memory >= 35 * 2**30, "Choose an A100 40 GB-class GPU."
assert shutil.which("nvcc"), "This runtime needs the CUDA toolkit."
if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")
RESULT_BASE.mkdir(parents=True, exist_ok=True)
LOCAL_BASE.mkdir(parents=True, exist_ok=True)
TORCH_VERSION = metadata.version("torch")
TORCH_FILE = str(Path(torch.__file__).resolve())
if SOURCE_MODE == "upload":
    if not SOURCE_ARCHIVE:
        uploaded = files.upload()
        assert len(uploaded) == 1, "Upload exactly one source ZIP."
        name, content = next(iter(uploaded.items()))
        assert name.lower().endswith(".zip"), "Select the source ZIP."
        archive_hash = hashlib.sha256(content).hexdigest()
        archive = RESULT_BASE / "source_uploads" / (archive_hash + ".zip")
        archive.parent.mkdir(parents=True, exist_ok=True)
        if archive.exists():
            assert file_sha256(archive) == archive_hash, "Saved source ZIP checksum differs."
        else:
            archive.write_bytes(content)
        SOURCE_ARCHIVE = str(archive)
        del uploaded, content
    REPO_DIR, SOURCE_ID = unpack_source_bundle(SOURCE_ARCHIVE, LOCAL_BASE / "sources")
    print("Saved ZIP for reconnect:", SOURCE_ARCHIVE)
else:
    assert re.fullmatch(r"[0-9a-fA-F]{40}", GIT_COMMIT), "Set a published full commit SHA."
    REPO_DIR = LOCAL_BASE / "sources" / GIT_COMMIT.lower()
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "--no-checkout", "https://github.com/CodeHalwell/rotquant.git", str(REPO_DIR)], check=True, timeout=300)
        subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", GIT_COMMIT], check=True, timeout=180)
        subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "--detach", GIT_COMMIT], check=True, timeout=60)
    assert subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip() == GIT_COMMIT.lower()
    assert not subprocess.check_output(["git", "-C", str(REPO_DIR), "status", "--porcelain"], text=True).strip(), "Source has changed; use a fresh directory."
    SOURCE_ID = GIT_COMMIT.lower()
for name in ("scripts/run_qwen35_ternary_research.py", "scripts/w4_research_measure.py", "rotquant/ternary.py", "rotquant/ternary_train.py", "scripts/ternary_research_bonsai.py", "scripts/ternary_bonsai_bridge.cpp", "configs/qwen35_4b_ternary_research_cuda.yaml"):
    assert (REPO_DIR / name).is_file(), f"Source is missing ternary research support: {name}"
sys.path.insert(0, str(REPO_DIR))
from scripts.colab_runtime import run_live as _run_live
SETUP_LOGS = RESULT_BASE / "setup" / SOURCE_ID[:16]
def run_live(command, label, timeout_seconds=None):
    return _run_live(command, label, repo_dir=REPO_DIR, log_root=SETUP_LOGS, timeout_seconds=timeout_seconds)
print({"source": SOURCE_ID, "gpu": torch.cuda.get_device_name(0), "results": str(RESULT_BASE)})

## Prepare the runtime
Dependencies live in a separate environment. The notebook checks that
Colab's exact CUDA PyTorch version and location remain unchanged. Setup
and fast-kernel build logs are saved to Drive.

In [ ]:
from urllib.request import urlopen
uv = shutil.which("uv")
if uv is None:
    tool_dir = LOCAL_BASE / "tools" / ("uv-" + UV_VERSION)
    tool_dir.mkdir(parents=True, exist_ok=True)
    installer = tool_dir / "install.sh"
    with urlopen(f"https://astral.sh/uv/{UV_VERSION}/install.sh", timeout=60) as response:
        installer.write_bytes(response.read())
    install_env = {**os.environ, "UV_INSTALL_DIR": str(tool_dir), "UV_NO_MODIFY_PATH": "1"}
    subprocess.run(["sh", str(installer)], env=install_env, check=True, timeout=180)
    uv = str(tool_dir / "uv")
print(subprocess.check_output([uv, "--version"], text=True).strip())
VENV = LOCAL_BASE / "environments" / SOURCE_ID[:16]
PYTHON = VENV / "bin/python"
if not PYTHON.exists():
    run_live([uv, "venv", "--python", sys.executable, "--system-site-packages", str(VENV)], "create-environment", 180)
check = [str(PYTHON), str(REPO_DIR / "scripts/native_gpu_environment.py"),
         "--prefix", str(VENV), "--expected-torch", TORCH_VERSION, "--expected-torch-file", TORCH_FILE]
run_live([*check, "--output", str(SETUP_LOGS / "torch-before.json")], "torch-before", 60)
constraints = VENV / "torch-constraint.txt"
constraints.write_text(f"torch=={TORCH_VERSION}\n")
packages = ["datasets==4.8.5", "accelerate==1.13.0", "pandas==2.3.3",
            "matplotlib==3.10.9", "seaborn==0.13.2", "huggingface_hub==1.17.0",
            "nltk==3.9.2", "langdetect==1.0.9", "immutabledict==4.2.1", "absl-py==2.3.1",
            "setuptools==80.9.0", "wheel==0.45.1"]
install = [sys.executable, "-m", "pip", "--isolated", "--python", str(PYTHON),
           "--require-virtualenv", "--disable-pip-version-check", "install",
           "--constraint", str(constraints)]
run_live([*install, "-r", str(REPO_DIR / "requirements/native-gpu.txt"), *packages], "dependencies", 1200)
run_live([*install, "--no-deps", "--no-build-isolation", "-e", str(REPO_DIR)], "install-project", 300)
run_live([*check, "--output", str(SETUP_LOGS / "torch-after.json")], "torch-after", 60)
os.environ.update({"MAX_JOBS": "2", "NVCC_THREADS": "2"})
smoke = ("import torch; from fast_hadamard_transform import hadamard_transform; "
         "x=hadamard_transform(torch.ones(1,128,device='cuda',dtype=torch.float16),scale=128**-0.5); "
         "assert torch.isfinite(x).all(); print('Fast Hadamard CUDA smoke passed')")
try:
    run_live([*install, "--no-deps", "--no-build-isolation",
              "git+https://github.com/Dao-AILab/fast-hadamard-transform.git@v1.1.0.post2"], "fast-hadamard-build", 1800)
    run_live([str(PYTHON), "-c", smoke], "fast-hadamard-smoke", 60)
    os.environ.pop("ROTQUANT_DISABLE_FAST_HADAMARD", None)
except (subprocess.CalledProcessError, TimeoutError):
    if REQUIRE_FAST_HADAMARD:
        raise
    os.environ["ROTQUANT_DISABLE_FAST_HADAMARD"] = "1"
run_live([*check, "--output", str(SETUP_LOGS / "torch-final.json")], "torch-final", 60)
if USE_HF_SECRET:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
    if token:
        os.environ["HF_TOKEN"] = token
    del token

## Review the plan
The driver checks native operators and whole-model graphs, prepares
fresh controls, compares ternary initialization and block QAT, and
tests separate recovery arms. Training, validation, selection and final
tests use separate data. The official Bonsai comparison uses this run's
inputs and measurements, never published scores copied into the table.

The default pilot uses ternary MLP matrices, W4 for the remaining
eligible backbone, W6 vocabulary and source-precision protected tensors.
The config also supports an all-backbone ternary experiment. Every row
records its precision scope. Canonical labels: `fp16`, `w6_v6`, `w5_v6`, `w4_v6`,
`w4_v6_recovered`, `ternary_ptq`, `ternary_blockqat`,
`ternary_recovered`, `bonsai2_4b`.

In [ ]:
config_reader = "import json,yaml,sys; print(json.dumps(yaml.safe_load(open(sys.argv[1]))))"
base_config = json.loads(subprocess.check_output([str(PYTHON), "-c", config_reader,
    str(REPO_DIR / "configs/qwen35_4b_ternary_research_cuda.yaml")], text=True))
RUNS = []
resume_parent = Path(RESUME_FROM).expanduser().resolve() if RESUME_FROM else None
resume_identity = None
assert type(FRESH_BUDGET) is bool and (not FRESH_BUDGET or resume_parent), "FRESH_BUDGET requires RESUME_FROM."
if resume_parent:
    assert len(SEEDS) == 1, "A continuation requires one seed."
    parent = json.loads((resume_parent / "identity.json").read_text())
    assert parent["config"]["seed"] == SEEDS[0], "The continuation seed must match its parent."
    resume_identity = {"root": str(resume_parent),
                       "identity": hashlib.sha256(json.dumps(parent, sort_keys=True).encode()).hexdigest()}
    if FRESH_BUDGET:
        resume_identity["fresh_budget"] = True
    print("Continuation parent:", json.dumps(resume_identity, indent=2))
assert not PARITY_CORRECTION or resume_parent, "A parity correction requires RESUME_FROM."
parity_correction = Path(PARITY_CORRECTION).expanduser().resolve() if PARITY_CORRECTION else None
if parity_correction:
    resume_identity["parity_correction"] = {"path": str(parity_correction),
        "sha256": hashlib.sha256(parity_correction.read_bytes()).hexdigest()}
minutes_per_seed = BUDGET_MINUTES // len(SEEDS)
for seed in SEEDS:
    config = json.loads(json.dumps(base_config))
    config["seed"] = seed
    config["ternary"]["quant"]["seed"] = seed
    config["research"]["public_samples_per_benchmark"] = PUBLIC_SAMPLES
    identity = {"study": STUDY_VERSION, "source": SOURCE_ID, "config": config, "budget_minutes": minutes_per_seed,
                "fast_hadamard_required": REQUIRE_FAST_HADAMARD}
    if resume_identity:
        identity["resume_from"] = resume_identity
    run_id = hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()[:16]
    root = RESULT_BASE / f"{run_id}-s{seed}"
    root.mkdir(parents=True, exist_ok=True)
    config_path = root / "config.yaml"
    text = json.dumps(config, indent=2) + "\n"  # JSON is also valid YAML.
    if config_path.exists():
        assert config_path.read_text() == text, "Stored run settings differ."
    else:
        config_path.write_text(text)
    (root / "notebook-identity.json").write_text(json.dumps(identity, indent=2) + "\n")
    command = [str(PYTHON), "-u", str(REPO_DIR / "scripts/run_qwen35_ternary_research.py"),
               "--config", str(config_path), "--output-dir", str(root),
               "--work-dir", str(LOCAL_BASE / "work" / run_id),
               "--budget-minutes", str(minutes_per_seed)]
    if resume_parent:
        command.extend(["--resume-from", str(resume_parent)])
    if FRESH_BUDGET:
        command.append("--fresh-budget")
    if parity_correction:
        command.extend(["--parity-correction", str(parity_correction)])
    RUNS.append({"seed": seed, "root": root, "command": command})
    _run_live([*command, "--dry-run"], "plan", repo_dir=REPO_DIR, log_root=root / "launch-logs", timeout_seconds=120)
print("Total execution allowance:", minutes_per_seed * len(SEEDS), "minutes")

## Run the comparison
Progress and logs stay on Drive. A failed gate stops that seed and leaves
its reports available below. Rerun with unchanged settings to reuse
verified completed stages; a lost Colab runtime may require local files
to be rebuilt. Avoid running two copies against the same result folder.

In [ ]:
if RUN_EXPERIMENT:
    for run in RUNS:
        try:
            _run_live(run["command"], f"research-s{run['seed']}", repo_dir=REPO_DIR,
                      log_root=run["root"] / "launch-logs")
        except (subprocess.CalledProcessError, TimeoutError) as error:
            print("This seed stopped:", run["seed"], str(error))
            print("Inspect its saved reports below:", run["root"])
else:
    print("Plan only; no model experiment was launched.")

## Results
These tables and plots use saved measurements from this run. Missing
values stay empty; status and error fields identify unfinished arms.
The size plots use **complete native payload bytes**, including retained
auxiliary tensors. Language-only and checkpoint sizes are separate.
Backend and precision scope stay visible; speed plots separate backends.

Task accuracy is shown as a percentage; paired deltas and intervals use
fractions. Counts and truncations matter for this small pilot. Timings
use frozen decode tokens and describe runtime speed, while public task
scores measure generated answers. Selection uses its dedicated data.
No row establishes unchanged capability by itself.
Completed quality/task measurements remain visible if timing fails;
that arm keeps its failed status. Resuming a fully completed run checks
its saved receipts without starting GPU work.

In [ ]:
PRESENTATION_SCRIPT = 'import json, math, sys\nfrom pathlib import Path\ndef measured_number(value):\n    return float(value) if isinstance(value, (int, float)) and not isinstance(value, bool) and math.isfinite(value) else None\n\n\ndef backend_label(row):\n    value = (row.get("speed") or {}).get("backend") or row.get("backend")\n    if isinstance(value, dict):\n        return json.dumps(value, sort_keys=True)\n    return str(value) if value else "Unreported backend"\n\n\ndef model_style(label):\n    labels = ["fp16", "w6_v6", "w5_v6", "w4_v6", "w4_v6_recovered", "ternary_ptq",\n              "ternary_blockqat", "ternary_recovered", "bonsai2_4b"]\n    index = labels.index(label) if label in labels else sum(map(ord, label))\n    colors = ["#494949", "#245ca6", "#a96919", "#526c38", "#9c4f77"]\n    markers = ["o", "s", "^", "D", "v", "P", "X", "<", ">"]\n    return colors[index % len(colors)], markers[index % len(markers)]\n\n\ndef model_comparison_row(row):\n    quality = row.get("quality") or {}\n    short = (quality.get("test") or {}).get("summary") or {}\n    long = (quality.get("long_test") or {}).get("summary") or {}\n    result = {"Model": row["label"], "Status": row.get("status", "unreported"),\n              "Backend": backend_label(row), "Precision scope": row.get("precision_scope"),\n              "Error": row.get("error"),\n              "Measurement origin": (row.get("origin") or {}).get("identity"),\n              "Origin results": (row.get("origin") or {}).get("root")}\n    for label, key in (("Checkpoint GB", "checkpoint_bytes"), ("Native GB", "native_payload_bytes"),\n                       ("Language GB", "language_payload_bytes")):\n        size = measured_number(row.get(key))\n        result[label] = None if size is None else size / 1e9\n    result.update({"Teacher KL": measured_number(short.get("mean_teacher_kl")),\n                   "Teacher KL 95% interval": (short.get("confidence_intervals") or {}).get("mean_teacher_kl"),\n                   "Perplexity": measured_number(short.get("ppl")),\n                   "Perplexity 95% interval": short.get("ppl_interval"),\n                   "Top-1 agreement": measured_number(short.get("top1_agreement")),\n                   "Long-context KL": measured_number(long.get("mean_teacher_kl")),\n                   "Test documents": short.get("documents"), "Test tokens": short.get("tokens")})\n    return result\n\n\ndef comparison_tables(report):\n    models, tasks, speed, paired = [], [], [], []\n    for row in report.get("rows", []):\n        models.append(model_comparison_row(row))\n        for benchmark, values in ((row.get("tasks") or {}).get("summary") or {}).items():\n            accuracy = measured_number(values.get("accuracy"))\n            tasks.append({"Model": row["label"], "Benchmark": benchmark,\n                          "Accuracy (%)": None if accuracy is None else 100 * accuracy,\n                          "Items": values.get("items"), "Correct": values.get("correct"),\n                          "Truncated": values.get("truncated"), "Backend": backend_label(row)})\n        speed_report = row.get("speed") or {}\n        for context, values in (speed_report.get("summary") or {}).items():\n            speed.append({"Model": row["label"], "Backend": backend_label(row),\n                          "Input tokens": int(context),\n                          "Prefill tokens/s": measured_number(values.get("prefill_tokens_per_second")),\n                          "Decode tokens/s": measured_number(values.get("decode_tokens_per_second")),\n                          "Repetitions": values.get("measured_repetitions"),\n                          "All contexts complete": speed_report.get("completed") is True})\n    for comparison, benchmarks in (report.get("paired_tasks") or {}).items():\n        for benchmark, values in benchmarks.items():\n            paired.append({"Comparison": comparison, "Benchmark": benchmark, "Items": values.get("items"),\n                           "Accuracy delta (fraction)": values.get("mean_accuracy_delta"),\n                           "Bootstrap 95% interval": values.get("bootstrap_interval"),\n                           "Hoeffding 95% interval": values.get("hoeffding_interval"),\n                           "Wins": values.get("candidate_wins"), "Losses": values.get("candidate_losses")})\n    return {"models": models, "tasks": tasks, "speed": speed, "paired": paired}\n\n\ndef figure_legend(figure, axes):\n    handles, labels = {}, []\n    for axis in axes:\n        for handle, label in zip(*axis.get_legend_handles_labels()):\n            if label not in handles:\n                labels.append(label)\n                handles[label] = handle\n    if labels:\n        figure.legend([handles[label] for label in labels], labels, loc="outside lower center",\n                      ncol=min(3, len(labels)), frameon=False)\n\n\ndef quality_points(axis, models, metric):\n    for row in models:\n        x, y = row["Native GB"], row[metric]\n        if x is None or y is None:\n            continue\n        color, marker = model_style(row["Model"])\n        axis.scatter(x, y, color=color, marker=marker, s=48, label=row["Model"])\n        interval = row.get(metric + " 95% interval")\n        if interval is not None:\n            axis.vlines(x, interval[0], interval[1], color=color, alpha=0.6)\n\n\ndef quality_figure(models):\n    import matplotlib.pyplot as plt\n\n    figure, axes = plt.subplots(1, 2, figsize=(13, 5.4), layout="constrained")\n    for axis, metric, title in zip(axes, ("Teacher KL", "Perplexity"),\n                                   ("Difference from FP16 · lower is better", "Held-out perplexity · lower is better")):\n        quality_points(axis, models, metric)\n        axis.set(xlabel="Measured complete native payload (GB)", ylabel=metric, title=title)\n        axis.grid(alpha=0.18)\n        axis.margins(x=0.22, y=0.25)\n    figure_legend(figure, axes)\n    figure.suptitle("Model size and quality · document-bootstrap 95% intervals", fontsize=15)\n    return figure\n\n\ndef task_points(axis, rows, sizes, benchmark):\n    for row in rows:\n        x, y = sizes.get(row["Model"]), row["Accuracy (%)"]\n        if row["Benchmark"] != benchmark or x is None or y is None:\n            continue\n        color, marker = model_style(row["Model"])\n        axis.scatter(x, y, color=color, marker=marker, s=48, label=row["Model"])\n\n\ndef task_figure(models, rows):\n    import matplotlib.pyplot as plt\n\n    sizes = {row["Model"]: row["Native GB"] for row in models}\n    benchmarks = list(dict.fromkeys(row["Benchmark"] for row in rows))\n    figure, axes = plt.subplots(1, len(benchmarks), figsize=(6 * len(benchmarks), 5.4),\n                                squeeze=False, layout="constrained")\n    for axis, benchmark in zip(axes[0], benchmarks):\n        task_points(axis, rows, sizes, benchmark)\n        axis.set(xlabel="Measured complete native payload (GB)", ylabel="Accuracy (%)",\n                 title=benchmark, ylim=(-2, 102))\n        axis.grid(alpha=0.18)\n        axis.margins(x=0.22)\n    figure_legend(figure, axes[0])\n    figure.suptitle("Public task quality and size\\nCounts and paired intervals in tables", fontsize=15)\n    return figure\n\n\ndef speed_lines(axis, rows, metric):\n    for label in dict.fromkeys(row["Model"] for row in rows):\n        values = sorted((row for row in rows if row["Model"] == label and row[metric] is not None),\n                        key=lambda row: row["Input tokens"])\n        if values:\n            color, marker = model_style(label)\n            axis.plot([row["Input tokens"] for row in values], [row[metric] for row in values],\n                      label=label, marker=marker, color=color)\n\n\ndef speed_axis(axis, rows, metric, backend):\n    speed_lines(axis, rows, metric)\n    maximum = max((row[metric] for row in rows if row[metric] is not None), default=0)\n    axis.set(xlabel="Input tokens", ylabel=metric, title=f"{backend}\\n{metric}",\n             ylim=(0, max(1, maximum * 1.15)))\n    axis.set_xticks(sorted({row["Input tokens"] for row in rows}))\n    axis.grid(alpha=0.18)\n    if axis.lines:\n        axis.legend(loc="best", fontsize=9, frameon=False)\n    else:\n        axis.text(0.5, 0.5, "Not measured", ha="center", va="center", transform=axis.transAxes)\n        axis.set_xticks([])\n        axis.set_yticks([])\n\n\ndef speed_figure(rows):\n    import matplotlib.pyplot as plt\n\n    backends = list(dict.fromkeys(row["Backend"] for row in rows))\n    figure, axes = plt.subplots(len(backends), 2, figsize=(13, 5.4 * len(backends)),\n                                squeeze=False, layout="constrained")\n    for pair, backend in zip(axes, backends):\n        subset = [row for row in rows if row["Backend"] == backend]\n        for axis, metric in zip(pair, ("Prefill tokens/s", "Decode tokens/s")):\n            speed_axis(axis, subset, metric, backend)\n    figure.suptitle("Measured runtime speed · medians, warmup excluded", fontsize=15)\n    return figure\n\n\ndef render_comparison(report, output):\n    import matplotlib\n    matplotlib.use("Agg")\n    import matplotlib.pyplot as plt\n\n    output = Path(output)\n    output.mkdir(parents=True, exist_ok=True)\n    tables = comparison_tables(report)\n    figures = []\n    if any(row["Native GB"] is not None and (row["Teacher KL"] is not None or row["Perplexity"] is not None)\n           for row in tables["models"]):\n        figures.append(("quality-size", quality_figure(tables["models"])))\n    sizes = {row["Model"]: row["Native GB"] for row in tables["models"]}\n    tasks = [row for row in tables["tasks"] if row["Accuracy (%)"] is not None\n             and sizes.get(row["Model"]) is not None]\n    if tasks:\n        figures.append(("tasks-size", task_figure(tables["models"], tasks)))\n    speed = [row for row in tables["speed"]\n             if row["Prefill tokens/s"] is not None or row["Decode tokens/s"] is not None]\n    if speed:\n        figures.append(("speed", speed_figure(speed)))\n    paths = []\n    for name, figure in figures:\n        path = output / f"{name}.png"\n        figure.savefig(path, dpi=160, facecolor="white")\n        plt.close(figure)\n        paths.append(str(path))\n    result = {"complete": report.get("complete") is True, "tables": tables, "figures": paths,\n              "selection": report.get("selection"), "error": report.get("error"),\n              "scope": report.get("scope"), "continuation": report.get("continuation")}\n    (output / "readout.json").write_text(json.dumps(result, indent=2) + "\\n")\n    return result\n\nif __name__ == "__main__":\n    render_comparison(json.loads(Path(sys.argv[1]).read_text()), sys.argv[2])\n'

In [ ]:
from html import escape
from IPython.display import HTML, Image, display
def display_table(title, rows):
    print(title)
    if not rows:
        print("Not measured yet.")
        return
    keys = list(rows[0])
    def value(item):
        if item is None:
            return "—"
        return f"{item:.6g}" if isinstance(item, float) else str(item)
    heading = "".join(f"<th>{escape(key)}</th>" for key in keys)
    body = "".join("<tr>" + "".join(f"<td>{escape(value(row.get(key)))}</td>" for key in keys) + "</tr>" for row in rows)
    display(HTML(f"<div style='overflow-x:auto'><table><thead><tr>{heading}</tr></thead><tbody>{body}</tbody></table></div>"))
for run in RUNS:
    report_path = run["root"] / "comparison.json"
    print("Seed", run["seed"], "·", run["root"])
    if not report_path.exists():
        print("Comparison not available yet. Read launch-logs and stage reports.")
        continue
    script = LOCAL_BASE / "render-ternary-results.py"
    script.write_text(PRESENTATION_SCRIPT)
    figures = run["root"] / "figures"
    _run_live([str(PYTHON), str(script), str(report_path), str(figures)], "render-results",
              repo_dir=REPO_DIR, log_root=run["root"] / "launch-logs", timeout_seconds=120)
    readout = json.loads((figures / "readout.json").read_text())
    print("Complete" if readout["complete"] else "Partial — some measurements are missing")
    if readout.get("error"):
        print("Stopped:", readout["error"])
    for title, key in (("Quality and measured bytes", "models"), ("Public tasks", "tasks"),
                       ("Measured speed by backend", "speed"), ("Paired task differences", "paired")):
        display_table(title, readout["tables"][key])
    for path in readout["figures"]:
        display(Image(filename=path))
    print("Scope:", readout.get("scope"))
    if readout.get("continuation"):
        print("Continuation provenance:", json.dumps(readout["continuation"], indent=2))
    print("Selection record:", json.dumps(readout["selection"], indent=2))

## Save the reports
The compact download contains reports, task outputs and figures. Full
checkpoints remain in Drive. When finished, disconnect
and delete the Colab runtime; the execution budget does not stop billing.

In [ ]:
archive = RESULT_BASE / ("ternary-v1-reports-" + SOURCE_ID[:16] + ".zip")
with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as output:
    for run in RUNS:
        for path in sorted(run["root"].rglob("*")):
            if (path.is_file() and path.suffix in {".json", ".log", ".csv", ".png", ".yaml"}
                    and path.stat().st_size <= 20 * 2**20
                    and not any(part in {"checkpoint", "teacher", "hessians"} for part in path.parts)):
                output.write(path, str(path.relative_to(RESULT_BASE)))
print("Reports:", archive)
files.download(str(archive))